# GTSRB: CNN-3 vs CNN-5 in TensorFlow and PyTorch

**Author:** Lưu Anh Dũng (B23DCDK036) · E23CNPM02 · Intelligent System Development · Assignment 05

**GTSRB** (German Traffic Sign Recognition Benchmark, IJCNN 2011) = 51,839 photos of traffic signs taken from a
moving car, resized to 32×32 color, **43 classes** (speed limits, stop, yield, priority road, …). Source: Kaggle
[`harbhajansingh21/german-traffic-sign-dataset`](https://www.kaggle.com/datasets/harbhajansingh21/german-traffic-sign-dataset).

What makes it hard: 43 classes, some with ~10× more images than others, very dark or blurred photos, and
look-alike signs (e.g. "speed limit 30" vs "80").

| Run ID | Framework | Model |
|---|---|---|
| `GT-TF-CNN-3` | TensorFlow/Keras | 3 convolutional layers |
| `GT-TF-CNN-5` | TensorFlow/Keras | 5 convolutional layers |
| `GT-PT-CNN-3` | PyTorch | 3 convolutional layers |
| `GT-PT-CNN-5` | PyTorch | 5 convolutional layers |

Sections 1–5 explain the method. Results are discussed in sections 6–7.

## 1. Setup

Seed 42 everywhere (same weights, batch order and dropout on every rerun). PyTorch device: `cuda` → `mps` → `cpu`;
TensorFlow finds a GPU by itself or uses the CPU. `A5_SMOKE=1` = quick test run.

In [ ]:
import copy
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


SEED = 42
warnings.filterwarnings("ignore", category=UserWarning)

import tensorflow as tf
import keras
from keras import layers

keras.utils.set_random_seed(SEED)

import torch
import torch.nn as nn

torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")

print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))
print("PyTorch", torch.__version__, "| device:", DEVICE)

**Helper functions** (no model code): project paths, the saved split, test metrics, result files and plots.
Settings shared by both frameworks: batch 256, max 20 epochs, early-stopping patience 3, learning rate 0.001.

In [ ]:
import json
import os
import random
from pathlib import Path

from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split

SMOKE = os.environ.get("A5_SMOKE") == "1"          # quick test: small subsample, 2 epochs


def find_root():
    """assignment_05/ = first folder above the working directory that has dataset/ and notebook/."""
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "dataset").is_dir() and (p / "notebook").is_dir():
            return p
    return here.parent


ROOT = find_root()
DATA_DIR = Path(os.environ.get("A5_DATA_DIR", ROOT / "dataset"))
RESULTS_DIR = ROOT / "notebook" / "results" / ("smoke" if SMOKE else "")
MODELS_DIR = ROOT / "notebook" / "models" / ("smoke" if SMOKE else "")
FIG_DIR = ROOT / "report" / "images" / ("smoke" if SMOKE else "")
for d in (RESULTS_DIR, MODELS_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 256
EPOCHS = 2 if SMOKE else 20
PATIENCE = 3
LEARNING_RATE = 1e-3
SMOKE_SIZE = {"train": 2000, "validation": 500, "test": 500}

random.seed(SEED)
np.random.seed(SEED)
print("project folder:", ROOT, "| smoke run:", SMOKE, "| epochs:", EPOCHS, "| batch size:", BATCH_SIZE)


def find_file(code, name):
    """Find `name` anywhere under dataset/<code>/ (unzipping may add a subfolder)."""
    base = DATA_DIR / code.lower()
    hits = sorted(base.rglob(name))
    if not hits:
        raise FileNotFoundError(f"{name} not found under {base}; see dataset/README.md")
    return hits[0]


def make_or_load_splits(code, y, n_sample=None):
    """Stratified 70/15/15 split of row indices, saved once to dataset/<code>/splits.npz."""
    path = DATA_DIR / code.lower() / "splits.npz"
    n_sample = min(n_sample or len(y), len(y))
    splits = None
    if path.exists():
        saved = np.load(path)
        splits = {k: saved[k] for k in ("train", "validation", "test")}
        if sum(len(v) for v in splits.values()) != n_sample:
            splits = None
    if splits is None:
        idx = np.arange(len(y))
        if n_sample < len(y):
            idx, _ = train_test_split(idx, train_size=n_sample, stratify=y, random_state=SEED)
        train, rest = train_test_split(idx, train_size=0.70, stratify=y[idx], random_state=SEED)
        val, test = train_test_split(rest, test_size=0.50, stratify=y[rest], random_state=SEED)
        splits = {"train": np.sort(train), "validation": np.sort(val), "test": np.sort(test)}
        np.savez(path, **splits)
    if SMOKE:
        rng = np.random.default_rng(SEED)
        splits = {k: np.sort(rng.choice(v, min(SMOKE_SIZE[k], len(v)), replace=False)) for k, v in splits.items()}
    return splits


def classification_metrics(y_true, y_pred, y_score=None, binary=False):
    """Accuracy + macro precision/recall/F1; for binary tasks also positive-class F1, ROC-AUC, AP."""
    m = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }
    if binary:
        m["f1_positive"] = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
        m["roc_auc"] = roc_auc_score(y_true, y_score)
        m["average_precision"] = average_precision_score(y_true, y_score)
    return {k: float(v) for k, v in m.items()}


class Stopwatch:
    def __enter__(self):
        self._start = time.perf_counter()
        return self

    def __exit__(self, *exc):
        self.seconds = time.perf_counter() - self._start


def save_result(record):
    (RESULTS_DIR / f"{record['run_id']}.json").write_text(json.dumps(record, indent=2), encoding="utf-8")


def load_results(run_ids):
    paths = [RESULTS_DIR / f"{rid}.json" for rid in run_ids]
    return [json.loads(p.read_text(encoding="utf-8")) for p in paths if p.exists()]


def savefig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")


def plot_history(history, run_id):
    """Figure with 2 plots: loss and accuracy per epoch, train vs validation."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    epochs = np.arange(1, len(history["loss"]) + 1)
    for ax, key, title in [(axes[0], "loss", "Loss"), (axes[1], "accuracy", "Accuracy")]:
        ax.plot(epochs, history[key], marker="o", label="train")
        ax.plot(epochs, history[f"val_{key}"], marker="o", label="validation")
        ax.set_title(f"{run_id}: {title}"); ax.set_xlabel("epoch"); ax.grid(alpha=0.3); ax.legend()
    fig.tight_layout(); savefig(fig, f"{run_id}_curves"); plt.show()


def plot_confusion(y_true, y_pred, labels, run_id):
    """Row-normalized confusion matrix: row = true class, column = predicted class."""
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(labels)))
    shown = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    many = len(labels) > 12
    size = max(4.5, 0.28 * len(labels))
    fig, ax = plt.subplots(figsize=(size + 1, size))
    im = ax.imshow(shown, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=90 if many else 0, fontsize=7 if many else 9)
    ax.set_yticklabels(labels, fontsize=7 if many else 9)
    if not many:
        for i in range(len(labels)):
            for j in range(len(labels)):
                ax.text(j, i, f"{shown[i, j]:.2f}", ha="center", va="center", fontsize=8,
                        color="white" if shown[i, j] > 0.5 else "black")
    ax.set_xlabel("predicted class"); ax.set_ylabel("true class")
    ax.set_title(f"{run_id}: confusion matrix (row-normalized)")
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout(); savefig(fig, f"{run_id}_confusion"); plt.show()
    return cm


def results_table(records):
    rows = []
    for r in records:
        row = {"run_id": r["run_id"], "framework": r["framework"], "model": r["model"], "params": r["params"],
               "epochs": r["epochs_run"], "time_per_epoch_s": r["time_per_epoch_s"],
               "train_time_s": r["train_time_s"], "inference_ms_per_1k": r["inference_ms_per_1k"]}
        row.update(r["test_metrics"])
        rows.append(row)
    return pd.DataFrame(rows).set_index("run_id")

Run IDs: `<dataset>-<framework>-<model>`, used for result files and figure names.

In [ ]:
DATASET = "GT"
MODELS = ["CNN-3", "CNN-5"]
FRAMEWORKS = ["TF", "PT"]
RUN_IDS = [f"{DATASET}-{fw}-{m}" for fw in FRAMEWORKS for m in MODELS]
RUN_IDS

## 2. Data

### 2.1 Load

Three Python pickle files (`train.p`, `valid.p`, `test.p`). Each is a dictionary:

| Key | Shape | Content |
|---|---|---|
| `features` | `(N, 32, 32, 3)` | images, `uint8` 0–255 (already height, width, color) |
| `labels` | `(N,)` | class id 0–42 |

`signname.csv` maps each class id to its name, e.g. `14 → Stop`.
Some copies were saved by Python 2; those need `encoding="latin1"` to open.

In [ ]:
import pickle


def load_pickle(name):
    with open(find_file(DATASET, name), "rb") as f:
        try:
            return pickle.load(f)
        except UnicodeDecodeError:          # saved by Python 2
            f.seek(0)
            return pickle.load(f, encoding="latin1")


parts = [load_pickle(name) for name in ["train.p", "valid.p", "test.p"]]
X_raw = np.concatenate([p["features"] for p in parts]).astype(np.uint8)    # (N, 32, 32, 3)
y_raw = np.concatenate([p["labels"] for p in parts]).astype(np.int64)

sign_names = pd.read_csv(find_file(DATASET, "signname.csv")).sort_values("ClassId")
CLASS_NAMES = sign_names["SignName"].tolist()
CLASS_LABELS = [str(i) for i in range(len(CLASS_NAMES))]     # plots use the id, names are in the table
print("X_raw:", X_raw.shape, "| y_raw:", y_raw.shape, "| classes:", len(CLASS_NAMES))
sign_names.set_index("ClassId").T

### 2.2 Split 70 / 15 / 15

The three files are pooled and split again, **stratified** by class. This matters here: a class with only
~250 images in total would get ~37 in the test set; without stratification it could get far fewer by chance.

train (70%) updates the weights · validation (15%) decides when to stop · test (15%) gives the final score.
Indices are saved in `dataset/gt/splits.npz`, shared by TensorFlow and PyTorch.

In [ ]:
splits = make_or_load_splits(DATASET, y_raw)
X_train, y_train = X_raw[splits["train"]], y_raw[splits["train"]]
X_val, y_val = X_raw[splits["validation"]], y_raw[splits["validation"]]
X_test, y_test = X_raw[splits["test"]], y_raw[splits["test"]]
N_CLASSES = len(CLASS_NAMES)

sizes = pd.Series({"train": len(y_train), "validation": len(y_val), "test": len(y_test)})
pd.DataFrame({"samples": sizes, "share": (sizes / sizes.sum()).round(3)})

### 2.3 Scale and layout

- `x / 255`: `0 → 0.0`, `255 → 1.0`, done per batch (images stay `uint8` in memory).
- Keras `(N, 32, 32, 3)`, PyTorch `(N, 3, 32, 32)`: same values, axes reordered.

In [ ]:
def to_channels_first(x):
    """(N, H, W, C) -> (N, C, H, W) for PyTorch; still uint8."""
    return np.ascontiguousarray(np.transpose(x, (0, 3, 1, 2)))

X_train_pt, X_val_pt, X_test_pt = (to_channels_first(x) for x in (X_train, X_val, X_test))
INPUT_SHAPE_TF = X_train.shape[1:]      # (H, W, C)
INPUT_CHANNELS = X_train.shape[-1]      # C
print("TF layout:", X_train.shape, "| PT layout:", X_train_pt.shape, "| dtype:", X_train.dtype)

## 3. Look at the data

**Images per class.** GTSRB is strongly **unbalanced** (common signs like speed limits have many more images
than rare warning signs). The cell prints the largest / smallest ratio.

Why it matters: accuracy counts images, so big classes dominate it. **Macro F1** averages the 43 per-class F1
scores with equal weight, so a rare sign that is always missed pulls it down (demo in 4.6).

In [ ]:
counts = pd.DataFrame({name: np.bincount(y_raw[idx], minlength=N_CLASSES) for name, idx in splits.items()},
                      index=CLASS_LABELS)
fig, ax = plt.subplots(figsize=(max(8, N_CLASSES * 0.3), 3.8))
counts.plot.bar(ax=ax, width=0.8)
ax.set_xlabel("class"); ax.set_ylabel("samples"); ax.set_title(f"{DATASET}: samples per class and split")
fig.tight_layout(); savefig(fig, f"{DATASET}_class_distribution"); plt.show()

train_counts = counts["train"]
print(f"smallest class: {train_counts.idxmin()} ({train_counts.min()}) | largest class: {train_counts.idxmax()} "
      f"({train_counts.max()}) | largest / smallest = {train_counts.max() / train_counts.min():.1f}")

**Samples.** One training image per class, titled `id: name`. Signs differ by **shape** (circle, triangle, octagon), **border color** and the **symbol** inside.

In [ ]:
cols = 9
rows = int(np.ceil(N_CLASSES / cols))
fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.5, rows * 1.75))
rng = np.random.default_rng(SEED)
for c, ax in enumerate(axes.ravel()):
    ax.set_xticks([]); ax.set_yticks([])
    if c >= N_CLASSES:
        ax.axis("off")
        continue
    ax.imshow(X_train[rng.choice(np.where(y_train == c)[0])])
    ax.set_title(f"{c}: {CLASS_NAMES[c][:16]}", fontsize=7)
fig.suptitle(f"{DATASET}: one training image per class", y=1.0)
fig.tight_layout(); savefig(fig, f"{DATASET}_samples"); plt.show()

**Pixel values and brightness.** Many GTSRB photos are very dark (dusk, shadow). The brightness histogram shows how wide that spread is; BatchNorm (4.2) is one answer to it.

In [ ]:
rng = np.random.default_rng(SEED)
sub = X_train[rng.choice(len(X_train), min(5000, len(X_train)), replace=False)].astype(np.float32) / 255.0
channel_names = ["red", "green", "blue"]
display(pd.DataFrame({"mean": sub.mean(axis=(0, 1, 2)), "std": sub.std(axis=(0, 1, 2))}, index=channel_names).round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ch, color in enumerate(["tab:red", "tab:green", "tab:blue"]):
    axes[0].hist(sub[..., ch].ravel(), bins=50, alpha=0.5, color=color, label=channel_names[ch])
axes[0].set_title(f"{DATASET}: pixel values per channel (train)"); axes[0].set_xlabel("value / 255"); axes[0].legend()
axes[1].hist(sub.mean(axis=(1, 2, 3)), bins=50, color="gray")
axes[1].set_title(f"{DATASET}: image brightness (train)"); axes[1].set_xlabel("mean pixel value / 255")
fig.tight_layout(); savefig(fig, f"{DATASET}_pixel_stats"); plt.show()

## 4. TensorFlow / Keras

### 4.1 Input pipeline

`tf.data`: shuffle (train only) → batch 256 → `/255` → prefetch. One batch: `(256, 32, 32, 3)` images, `(256,)` labels.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def scale(x, y):
    return tf.cast(x, tf.float32) / 255.0, y


def make_tf_dataset(x, y, training):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(len(x), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).map(scale, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)


train_ds = make_tf_dataset(X_train, y_train, training=True)
val_ds = make_tf_dataset(X_val, y_val, training=False)
test_ds = make_tf_dataset(X_test, y_test, training=False)
train_ds.element_spec

### 4.2 One conv block on a sign edge

A conv block is **Conv → BatchNorm → ReLU → MaxPool**. Example: the diagonal side of a triangular warning sign,
as a 5×5 patch (1 = sign, 0 = background), and a diagonal-edge filter:

```
patch          filter
1 0 0 0 0       0 -1 -1
1 1 0 0 0       1  0 -1
1 1 1 0 0       1  1  0
1 1 1 1 0
1 1 1 1 1
```

- **Convolution** `out = Σ (window × filter) + bias`:
  - window on the diagonal `[[1,0,0],[1,1,0],[1,1,1]]` → row sums `0 + 1 + 2 = 3`: strong response
  - window inside the sign (all 1s) → sum of the filter = `0`: no response
- **ReLU** `= max(0, x)`: keeps `3`, turns negatives into `0`.
- **MaxPool 2×2**: largest of 4 values, e.g. `[[3, 0], [3, 3]] → 3`. The edge is still "seen" at half the resolution.
- **BatchNorm** per channel over the batch: `(x − μ) / σ`, then `γ·x̂ + β`.
  A batch mixing dark and bright pixels `[0.05, 0.10, 0.80, 0.90]` (μ = 0.46, σ = 0.39) becomes
  `[−1.06, −0.93, 0.87, 1.12]`: the next layer always sees values around 0 with spread 1, whatever the lighting.

The cell repeats this with NumPy and checks against `tf.nn.conv2d`.

In [ ]:
patch = np.tril(np.ones((5, 5), dtype=np.float32))                   # diagonal edge of a triangle
kernel = np.array([[0, -1, -1], [1, 0, -1], [1, 1, 0]], dtype=np.float32)

conv = np.array([[np.sum(patch[i:i + 3, j:j + 3] * kernel) for j in range(3)] for i in range(3)])
tf_conv = tf.nn.conv2d(patch[None, :, :, None], kernel[:, :, None, None], strides=1, padding="VALID")[0, :, :, 0]
print("convolution (by hand):\n", conv)
print("same as tf.nn.conv2d:", np.allclose(conv, tf_conv.numpy()))
print("ReLU:\n", np.maximum(0, conv))

batch = np.array([0.05, 0.10, 0.80, 0.90])
print("BatchNorm of", batch, "->", ((batch - batch.mean()) / np.sqrt(batch.var() + 1e-5)).round(2))

### 4.3 The two models

3×3 filters, `padding="same"` (32×32 stays 32×32 before pooling). CNN-5 adds a second convolution before the
first two poolings.

| Step | CNN-3 output | CNN-5 output |
|---|---|---|
| input | 32×32×3 | 32×32×3 |
| block 1 | Conv 32 → Pool → **16×16×32** | Conv 32 → Conv 32 → Pool → **16×16×32** |
| block 2 | Conv 64 → Pool → **8×8×64** | Conv 64 → Conv 64 → Pool → **8×8×64** |
| block 3 | Conv 128 → Pool → **4×4×128** | Conv 128 → Pool → **4×4×128** |
| head | GAP → Dropout 0.3 → Dense 256 → **Dense 43** | same |

- Layer 1 finds edges and colors (red border, white inside); deeper layers combine them into shapes and symbols.
- **Receptive field:** one final unit sees 22×22 pixels in CNN-3 and 28×28 in CNN-5. A GTSRB sign fills most of
  the 32×32 image, so CNN-5 units see almost the whole sign at once.
- **GAP** = mean of each 4×4 map → 128 numbers. **Dropout 0.3** zeroes 30% of them during training.
- **Dense 43** = one score per sign class.

The second cell sends one real sign through every layer and prints the shapes.

In [ ]:
ARCH = {  # (filters, max-pool after this conv?)
    "CNN-3": [(32, True), (64, True), (128, True)],
    "CNN-5": [(32, False), (32, True), (64, False), (64, True), (128, True)],
}
HEAD_UNITS = 256
DROPOUT = 0.3


def conv_block_tf(x, filters, pool):
    x = layers.Conv2D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization(momentum=0.9, epsilon=1e-5)(x)
    x = layers.ReLU()(x)
    if pool:
        x = layers.MaxPooling2D(2)(x)
    return x


def build_tf(model_name, input_shape, n_classes):
    inputs = keras.Input(shape=input_shape)
    x = inputs
    for filters, pool in ARCH[model_name]:
        x = conv_block_tf(x, filters, pool)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(DROPOUT)(x)
    x = layers.Dense(HEAD_UNITS, activation="relu")(x)
    outputs = layers.Dense(n_classes)(x)            # logits
    return keras.Model(inputs, outputs, name=f"{DATASET}_TF_{model_name.replace('-', '')}")

In [ ]:
image = X_train[:1].astype(np.float32) / 255.0          # one training image, batch of 1
for m in MODELS:
    model = build_tf(m, INPUT_SHAPE_TF, N_CLASSES)
    x, rows = image, []
    for layer in model.layers[1:]:
        x = layer(x, training=False)
        rows.append((layer.__class__.__name__, tuple(x.shape[1:]), layer.count_params()))
    print(f"{m}: one image through every layer")
    display(pd.DataFrame(rows, columns=["layer", "output shape", "params"]).T)

### 4.4 Parameters

- Conv: `(K·K·C_in + 1)·F`, e.g. first layer `(3·3·3 + 1)·32 = 896`.
- BatchNorm: `2·F` trainable (γ, β).
- Dense: `(inputs + 1)·units`. The 43-class output alone has `(256 + 1)·43 = 11,051`.
- Pooling, ReLU, dropout: none.

The PyTorch models must match these totals (section 5).

In [ ]:
def expected_params(model_name, in_channels, n_classes):
    rows, c = [], in_channels
    for i, (f, _) in enumerate(ARCH[model_name], start=1):
        rows.append((f"conv{i}", f"(3·3·{c} + 1)·{f}", (9 * c + 1) * f))
        rows.append((f"bn{i}", f"2·{f}", 2 * f))
        c = f
    rows.append(("dense", f"({c} + 1)·{HEAD_UNITS}", (c + 1) * HEAD_UNITS))
    rows.append(("output", f"({HEAD_UNITS} + 1)·{n_classes}", (HEAD_UNITS + 1) * n_classes))
    return pd.DataFrame(rows, columns=["layer", "formula", "params"])


def trainable_params_tf(model):
    return int(sum(np.prod(w.shape) for w in model.trainable_weights))


for m in MODELS:
    table = expected_params(m, INPUT_CHANNELS, N_CLASSES)
    print(f"{m}: {table.params.sum():,} trainable parameters")
    display(table.set_index("layer").T)

### 4.5 Loss and training

- **Softmax over 43 classes:** `p_k = e^(z_k) / Σ_j e^(z_j)`, so the 43 probabilities sum to 1.
- **Cross-entropy** = `−ln p(true class)`.
  - Untrained model: `p = 1/43 = 0.023` for every class → loss `= ln 43 = 3.76` at the start.
  - Confident and right, `p = 0.95` → loss `0.05`. Confident and wrong, `p(true) = 0.01` → loss `4.6`.
- **Adam**, learning rate 0.001, batch 256 (≈ 142 updates per epoch on 36k training images).
- **Early stopping:** stop after 3 epochs without a lower validation loss, keep the best weights; max 20 epochs.
- **Time per epoch** = median of epochs 2+ (epoch 1 includes setup).

In [ ]:
class EpochTimer(keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.times = []

    def on_epoch_begin(self, epoch, logs=None):
        self._start = time.perf_counter()

    def on_epoch_end(self, epoch, logs=None):
        self.times.append(time.perf_counter() - self._start)


def train_tf(model):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE, epsilon=1e-7),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"],
    )
    timer = EpochTimer()
    stopper = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)
    history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=[stopper, timer], verbose=2)
    hist = {k: [float(v) for v in vals] for k, vals in history.history.items()}
    return hist, timer.times, int(np.argmin(hist["val_loss"])) + 1

### 4.6 Test metrics: why macro F1

Per class: **precision** = TP / (TP + FP), **recall** = TP / (TP + FN), **F1** = 2PR / (P + R).
**Macro F1** = plain mean of the 43 F1 scores.

Toy example with an unbalanced test set: 95 "speed limit" images and 5 "rare sign" images. A model that always
answers "speed limit" gets **accuracy 0.95** but **macro F1 0.49** (F1 = 0.97 for the big class, 0 for the rare one).
The cell below computes it. On GTSRB, macro F1 shows whether rare signs are recognized too.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

toy_true = np.array([0] * 95 + [1] * 5)
toy_pred = np.zeros(100, dtype=int)                       # always predicts the big class
print("accuracy:", accuracy_score(toy_true, toy_pred),
      "| per-class F1:", f1_score(toy_true, toy_pred, average=None, zero_division=0).round(3),
      "| macro F1:", round(f1_score(toy_true, toy_pred, average="macro", zero_division=0), 3))

In [ ]:
def evaluate_tf(model):
    model.predict(test_ds.take(1), verbose=0)           # warm-up, not timed
    with Stopwatch() as t:
        logits = model.predict(test_ds, verbose=0)
    probs = tf.nn.softmax(logits).numpy()
    y_pred = probs.argmax(axis=1)
    return y_pred, probs, classification_metrics(y_test, y_pred), 1000 * t.seconds / len(y_test)


def make_record(run_id, framework, model_name, params, hist, epoch_times, best_epoch, metrics, ms_per_1k):
    return {
        "run_id": run_id, "dataset": DATASET, "framework": framework, "model": model_name,
        "params": params, "epochs_run": len(epoch_times), "best_epoch": best_epoch,
        "train_time_s": float(sum(epoch_times)),
        # epoch 1 includes one-time setup (graph tracing, GPU kernel loading): excluded here
        "time_per_epoch_s": float(np.median(epoch_times[1:] if len(epoch_times) > 1 else epoch_times)),
        "inference_ms_per_1k": float(ms_per_1k), "test_metrics": metrics,
        "history": {k: hist[k] for k in ("loss", "val_loss", "accuracy", "val_accuracy")},
        "smoke": SMOKE,
    }


def run_tf(model_name):
    run_id = f"{DATASET}-TF-{model_name}"
    keras.utils.set_random_seed(SEED)                 # same initial weights on every rerun
    model = build_tf(model_name, INPUT_SHAPE_TF, N_CLASSES)
    params = trainable_params_tf(model)
    assert params == expected_params(model_name, INPUT_CHANNELS, N_CLASSES).params.sum()
    model.summary()
    hist, epoch_times, best_epoch = train_tf(model)
    y_pred, probs, metrics, ms_per_1k = evaluate_tf(model)
    record = make_record(run_id, "TF", model_name, params, hist, epoch_times, best_epoch, metrics, ms_per_1k)
    save_result(record)
    model.save(MODELS_DIR / f"{run_id}.keras")
    print(pd.Series(metrics).round(4).to_string())
    plot_history(record["history"], run_id)
    plot_confusion(y_test, y_pred, CLASS_LABELS, run_id)
    return model, y_pred, record

### 4.7 `GT-TF-CNN-3`
Build → check parameters → train → test → save `results/GT-TF-CNN-3.json` → curves and 43×43 confusion matrix (row = true sign, column = predicted).

In [ ]:
tf_models, tf_preds = {}, {}
tf_models["CNN-3"], tf_preds["CNN-3"], _ = run_tf("CNN-3")

### 4.8 `GT-TF-CNN-5`

In [ ]:
tf_models["CNN-5"], tf_preds["CNN-5"], _ = run_tf("CNN-5")

### 4.9 What the first layer learned

- **Filters:** the 32 first-layer filters (3×3×3), drawn as tiny color images. Red/white contrast filters are
  typical for sign borders.
- **Feature maps:** each filter's output after ReLU for one test sign; bright = strong response.

In [ ]:
sample = X_test[:1].astype(np.float32) / 255.0          # one test image


def show_filters(filters, run_id):
    """filters: (F, K, K, C_in). Each filter is min-max scaled and drawn as a 3x3 color image."""
    fig, axes = plt.subplots(4, 8, figsize=(8, 4.4))
    for ax in axes.ravel():
        ax.axis("off")
    for ax, f in zip(axes.ravel(), filters):
        ax.imshow((f - f.min()) / (f.max() - f.min() + 1e-8))
    fig.suptitle(f"{run_id}: the 32 first-layer filters (3x3x3)")
    fig.tight_layout(); savefig(fig, f"{run_id}_filters"); plt.show()


def show_feature_maps(maps, run_id):
    """maps: (F, H, W) after the first ReLU. First panel = the input image."""
    fig, axes = plt.subplots(2, 8, figsize=(12, 3.6))
    for ax in axes.ravel():
        ax.axis("off")
    axes[0, 0].imshow(sample[0]); axes[0, 0].set_title("input", fontsize=9)
    for ax, fm in zip(axes.ravel()[1:], maps):
        ax.imshow(fm, cmap="gray")
    fig.suptitle(f"{run_id}: input and 15 feature maps after the first ReLU")
    fig.tight_layout(); savefig(fig, f"{run_id}_feature_maps"); plt.show()


def tf_filters(m):
    conv = next(l for l in tf_models[m].layers if isinstance(l, layers.Conv2D))
    return np.transpose(conv.get_weights()[0], (3, 0, 1, 2))            # (K, K, C_in, F) -> (F, K, K, C_in)


def tf_maps(m):
    model = tf_models[m]
    relu = next(l for l in model.layers if isinstance(l, layers.ReLU))
    return np.transpose(keras.Model(model.inputs, relu.output)(sample).numpy()[0], (2, 0, 1))


for m in MODELS:
    show_filters(tf_filters(m), f"{DATASET}-TF-{m}")
    show_feature_maps(tf_maps(m), f"{DATASET}-TF-{m}")

### 4.10 Errors
Test signs CNN-5 got wrong (true / predicted class id; names in the table in 2.1).

In [ ]:
def show_misclassified(y_pred, run_id, n=16):
    wrong = np.where(y_pred != y_test)[0][:n]
    cols = 8
    rows = max(1, int(np.ceil(len(wrong) / cols)))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.6, rows * 1.9), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, i in zip(axes.ravel(), wrong):
        ax.imshow(X_test[i])
        ax.set_title(f"true {CLASS_LABELS[y_test[i]]}\npred {CLASS_LABELS[y_pred[i]]}", fontsize=8)
    fig.suptitle(f"{run_id}: misclassified test images", y=1.02)
    fig.tight_layout(); savefig(fig, f"{run_id}_misclassified"); plt.show()


show_misclassified(tf_preds["CNN-5"], f"{DATASET}-TF-CNN-5")

## 5. PyTorch

Same models, data and settings, written differently:

| Step | Keras | PyTorch |
|---|---|---|
| data | `tf.data` | `TensorDataset` + `DataLoader` |
| layout | `(N, 32, 32, 3)` | `(N, 3, 32, 32)` |
| conv | `Conv2D(32, 3, padding="same")` | `Conv2d(3, 32, 3, padding=1)` (input channels explicit) |
| output | `Dense(43)` | `Linear(256, 43)` (input size explicit) |
| initial weights | Glorot uniform | `xavier_uniform_` (= Glorot), set by hand |
| BatchNorm momentum | 0.9 (old value) | 0.1 (new value): same update |
| training | `fit` | hand-written loop |

### 5.1 Data loader

In [ ]:
from torch.utils.data import DataLoader, TensorDataset


def make_loader(x, y, training):
    ds = TensorDataset(torch.from_numpy(x), torch.from_numpy(y))
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=training, generator=g if training else None)


train_loader = make_loader(X_train_pt, y_train, training=True)
val_loader = make_loader(X_val_pt, y_val, training=False)
test_loader = make_loader(X_test_pt, y_test, training=False)


def to_device(x):
    return x.to(DEVICE).float() / 255.0


xb, yb = next(iter(train_loader))
print("batch:", tuple(xb.shape), xb.dtype, "| labels:", tuple(yb.shape))

### 5.2 Model class
`__init__` creates the layers, `forward` runs them. The check below confirms the same parameter count as Keras.

In [ ]:
def init_like_keras(module):
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        nn.init.xavier_uniform_(module.weight)
        nn.init.zeros_(module.bias)


class CNN(nn.Module):
    def __init__(self, model_name, in_channels, n_classes):
        super().__init__()
        blocks, c = [], in_channels
        for filters, pool in ARCH[model_name]:
            blocks += [nn.Conv2d(c, filters, kernel_size=3, padding=1),
                       nn.BatchNorm2d(filters, momentum=0.1, eps=1e-5),
                       nn.ReLU()]
            if pool:
                blocks.append(nn.MaxPool2d(2))
            c = filters
        self.features = nn.Sequential(*blocks)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(DROPOUT),
            nn.Linear(c, HEAD_UNITS), nn.ReLU(),
            nn.Linear(HEAD_UNITS, n_classes),       # logits
        )
        self.apply(init_like_keras)

    def forward(self, x):
        return self.head(self.features(x))


def trainable_params_pt(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


for m in MODELS:
    n_pt = trainable_params_pt(CNN(m, INPUT_CHANNELS, N_CLASSES))
    n_formula = expected_params(m, INPUT_CHANNELS, N_CLASSES).params.sum()
    assert n_pt == n_formula, (m, n_pt, n_formula)
    print(f"{m}: PyTorch {n_pt:,} = formula {n_formula:,} = Keras (match)")

### 5.3 Training loop

Per batch: `zero_grad()` (clear gradients) → `model(x)` (forward, `(256, 3, 32, 32) → (256, 43)`) →
`CrossEntropyLoss` (same formula as 4.5) → `backward()` (gradients) → `step()` (Adam update).

`model.train()` = dropout on, BatchNorm uses batch statistics. `model.eval()` + `no_grad()` = dropout off,
BatchNorm uses running statistics, no gradients. Early stopping is a counter: 3 epochs without improvement → stop,
reload the best weights.

In [ ]:
def sync():
    if DEVICE.type == "mps":
        torch.mps.synchronize()
    elif DEVICE.type == "cuda":
        torch.cuda.synchronize()


@torch.no_grad()
def evaluate_loader(model, loader, criterion):
    model.eval()
    total_loss, correct, n, logits_all = 0.0, 0, 0, []
    for x, y in loader:
        x, y = to_device(x), y.to(DEVICE)
        logits = model(x)
        total_loss += criterion(logits, y).item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        n += len(y)
        logits_all.append(logits.cpu())
    return total_loss / n, correct / n, torch.cat(logits_all)


def train_pt(model):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, eps=1e-7)
    hist = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}
    epoch_times, best_loss, best_state, wait = [], float("inf"), None, 0

    for epoch in range(1, EPOCHS + 1):
        sync(); start = time.perf_counter()
        model.train()
        total_loss, correct, n = 0.0, 0, 0
        for x, y in train_loader:
            x, y = to_device(x), y.to(DEVICE)
            optimizer.zero_grad()              # 1. clear old gradients
            logits = model(x)                  # 2. forward pass
            loss = criterion(logits, y)        # 3. loss
            loss.backward()                    # 4. backpropagation
            optimizer.step()                   # 5. Adam update
            total_loss += loss.item() * len(y)
            correct += (logits.argmax(1) == y).sum().item()
            n += len(y)
        val_loss, val_acc, _ = evaluate_loader(model, val_loader, criterion)
        sync(); epoch_times.append(time.perf_counter() - start)

        for k, v in zip(hist, (total_loss / n, correct / n, val_loss, val_acc)):
            hist[k].append(float(v))
        print(f"Epoch {epoch}/{EPOCHS} - {epoch_times[-1]:.1f}s - loss: {hist['loss'][-1]:.4f} - "
              f"accuracy: {hist['accuracy'][-1]:.4f} - val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}")

        if val_loss < best_loss:               # early stopping, written by hand
            best_loss, best_state, wait = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f"Epoch {epoch}: early stopping")
                break

    model.load_state_dict(best_state)
    best_epoch = int(np.argmin(hist["val_loss"])) + 1
    print(f"Restoring model weights from the end of the best epoch: {best_epoch}.")
    return hist, epoch_times, best_epoch


def evaluate_pt(model):
    criterion = nn.CrossEntropyLoss()
    evaluate_loader(model, [next(iter(test_loader))], criterion)       # warm-up
    sync()
    with Stopwatch() as t:
        _, _, logits = evaluate_loader(model, test_loader, criterion)
        sync()
    probs = torch.softmax(logits, dim=1).numpy()
    y_pred = probs.argmax(axis=1)
    return y_pred, probs, classification_metrics(y_test, y_pred), 1000 * t.seconds / len(y_test)


def run_pt(model_name):
    run_id = f"{DATASET}-PT-{model_name}"
    torch.manual_seed(SEED)
    model = CNN(model_name, INPUT_CHANNELS, N_CLASSES).to(DEVICE)
    params = trainable_params_pt(model)
    print(model)
    print(f"trainable parameters: {params:,}")
    hist, epoch_times, best_epoch = train_pt(model)
    y_pred, probs, metrics, ms_per_1k = evaluate_pt(model)
    record = make_record(run_id, "PT", model_name, params, hist, epoch_times, best_epoch, metrics, ms_per_1k)
    save_result(record)
    torch.save(model.state_dict(), MODELS_DIR / f"{run_id}.pt")
    print(pd.Series(metrics).round(4).to_string())
    plot_history(record["history"], run_id)
    plot_confusion(y_test, y_pred, CLASS_LABELS, run_id)
    return model, y_pred, record

### 5.4 `GT-PT-CNN-3`

In [ ]:
pt_models, pt_preds = {}, {}
pt_models["CNN-3"], pt_preds["CNN-3"], _ = run_pt("CNN-3")

### 5.5 `GT-PT-CNN-5`

In [ ]:
pt_models["CNN-5"], pt_preds["CNN-5"], _ = run_pt("CNN-5")

### 5.6 What the first layer learned
Filters come from `features[0].weight` `(32, 3, 3, 3)`; feature maps are saved by a **forward hook** on the first ReLU.

In [ ]:
def pt_filters(m):
    return pt_models[m].features[0].weight.detach().cpu().numpy().transpose(0, 2, 3, 1)   # (F, K, K, C_in)


def pt_maps(m):
    model, captured = pt_models[m].eval(), {}
    relu = next(l for l in model.features if isinstance(l, nn.ReLU))
    handle = relu.register_forward_hook(lambda mod, inp, out: captured.update(out=out))
    with torch.no_grad():
        model(torch.from_numpy(np.transpose(sample, (0, 3, 1, 2))).to(DEVICE))
    handle.remove()
    return captured["out"][0].cpu().numpy()                                               # (F, H, W)


for m in MODELS:
    show_filters(pt_filters(m), f"{DATASET}-PT-{m}")
    show_feature_maps(pt_maps(m), f"{DATASET}-PT-{m}")

### 5.7 Errors

In [ ]:
show_misclassified(pt_preds["CNN-5"], f"{DATASET}-PT-CNN-5")

## 6. Comparison

Read from the four `results/<run_id>.json` files. Macro F1 is the key column here because of the class imbalance.

In [ ]:
records = load_results(RUN_IDS)
comparison = results_table(records)
comparison.to_csv(RESULTS_DIR / f"{DATASET}_comparison.csv")
comparison.round(4)

Charts (two per figure) and the differences **CNN-5 − CNN-3** and **PT − TF**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
comparison[["accuracy", "f1_macro"]].plot.bar(ax=axes[0], rot=0)
axes[0].set_ylim(max(0, comparison[["accuracy", "f1_macro"]].min().min() - 0.1), 1)
axes[0].set_title(f"{DATASET}: test accuracy and macro F1"); axes[0].grid(axis="y", alpha=0.3)
comparison["time_per_epoch_s"].plot.bar(ax=axes[1], rot=0, color="gray")
axes[1].set_title(f"{DATASET}: training time per epoch (s)"); axes[1].grid(axis="y", alpha=0.3)
fig.tight_layout(); savefig(fig, f"{DATASET}_comparison_bars"); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for r in records:
    style = "-" if r["framework"] == "TF" else "--"
    ep = np.arange(1, len(r["history"]["val_loss"]) + 1)
    axes[0].plot(ep, r["history"]["val_loss"], style, marker="o", label=r["run_id"])
    axes[1].plot(ep, r["history"]["val_accuracy"], style, marker="o", label=r["run_id"])
axes[0].set_title(f"{DATASET}: validation loss"); axes[1].set_title(f"{DATASET}: validation accuracy")
for ax in axes:
    ax.set_xlabel("epoch"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
fig.tight_layout(); savefig(fig, f"{DATASET}_comparison_curves"); plt.show()

cols = ["accuracy", "f1_macro", "params", "time_per_epoch_s", "epochs"]
by = comparison.set_index(["framework", "model"])[cols]
print("CNN-5 minus CNN-3 (per framework):")
display(pd.DataFrame({fw: by.loc[(fw, "CNN-5")] - by.loc[(fw, "CNN-3")] for fw in FRAMEWORKS}).T.round(4))
print("PT minus TF (per model):")
display(pd.DataFrame({m: by.loc[("PT", m)] - by.loc[("TF", m)] for m in MODELS}).T.round(4))

**Discussion** *(write after the full run)*
- Does CNN-5 improve macro F1 more than accuracy (i.e. does depth help the rare signs)?
- Do TF and PT agree? Which is faster?
- Which signs are confused (look-alike speed limits? dark images)?

## 7. Conclusion

*(3–5 takeaways, written after the full run)*
1. …
2. …
3. …